In [21]:
import pandas as pd
import geopandas as gpd

In [29]:
oak_parcels = gpd.read_parquet("data/processed/oak_parcels_sb79tiers_zones.parquet").copy()

#import zoning and SB79 description
#the zoning data was compiled into a csv with the help of ChatGPT and then manually verified
oak_zoning = pd.read_csv("data/raw/cyplan101_zoning_oak.csv")
sb79_df = pd.read_csv("data/raw/cyplan101_sb79.csv")

In [30]:
#merge use SB79 information onto parcel data
oak_parcels = oak_parcels.merge(
    sb79_df,
    left_on="tier_zone_label",
    right_on="tier_zone_label",
    how="left",
    validate="many_to_one"
)

#merge zoning information onto parcel data
oak_parcels = oak_parcels.merge(
    oak_zoning[['input_label', 'zoning_description', 'type', 'group_scale', 'residential_permission_notes', 'max_height_ft_clean']],
    left_on="basezone",
    right_on="input_label",
    how="left",
    validate="many_to_one"
)

In [33]:
#ensure the scale is numeric
oak_parcels['sb79_scale'] = pd.to_numeric(oak_parcels['sb79_scale'], errors='coerce')

oak_parcels['group_scale'] = pd.to_numeric(oak_parcels['group_scale'], errors='coerce')

In [34]:
#if the zone type is not eligible for upzoning, set the SB79 scale to None
mask = ~oak_parcels['type'].isin(['commercial', 'mixed_use', 'residential'])
oak_parcels.loc[mask, ['sb79_scale', 'max_height_sb79']] = None

In [35]:
#save cleaned and joined gdf as a parqet file to use for analysis
oak_parcels.to_parquet("data/processed/oak_parcels.parquet")